# Price Analysis - Casas de California

Notebook sobre el analisis realizado al dataset de `data/housing.csv` (20.640 distritos de California, de Kaggle).

Estructura:
1. Importacion de librerías
2. Carga de datos
3. Exploración inicial
4. Limpieza / preprocesamiento
5. Análisis simple
6. Análisis complejo

# Sección 1 - Importacion de librerías

pandas (tablas), numpy (matemática), matplotlib y seaborn (gráficos).
`sns.set_theme()` aplica estilo y `%matplotlib inline` muestra los gráficos dentro del notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt # graficos
import seaborn as sns # // 

sns.set_theme() 

%matplotlib inline

# Sección 2 - Carga de datos

El notebook está en `notebooks/`, por eso la ruta sube un nivel: `../data/housing.csv`.

In [2]:
df = pd.read_csv("../data/housing.csv")

df.shape

(20640, 10)

# Sección 3 - Exploración inicial

Qué se revisó:
- Primeras filas con `.head()`.
- Dimensiones: 20.640 filas y 10 columnas.
- Tipos de datos: 9 columnas numéricas (float64) y `ocean_proximity` como texto.
- Nulos: solo `total_bedrooms` tiene 207.
- Resumen estadístico con `.describe()`.
- Duplicados: no hay.

In [3]:
estructure = df.head()
count = df.shape  #propiedades                
columns = df.columns                
types = df.dtypes
nulls = df.isnull().sum()  
resume = df.describe()
duplicates = df.duplicated().sum()

display(estructure, count, columns, types, nulls, resume, duplicates)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


(20640, 10)

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='str')

longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity           str
dtype: object

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


np.int64(0)

# Sección 4 - Limpieza / preprocesamiento

 Decisión: nulos de `total_bedrooms`

Se hallaron un total de 207 nulos (~1%) en la columna de 'total_bedrooms', para preservar esas entradas del dataset se relleno tales faltas con la mediana, en lugar de borrar las filas.

Luego se verificó que no quedaran nulos y se revisaron las categorías de
`ocean_proximity`.

In [4]:


median = df['total_bedrooms'].median()

df['total_bedrooms'] = df['total_bedrooms'].fillna(median) # reemplazo de valores nulos

display(df.isnull().sum()) # verificacion

df['ocean_proximity'].value_counts() # categorias 

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

# Sección 4b - Feature Engineering (opcional)

Creamos ratios que se usan en la industria inmobiliaria. Estos ratios
correlacionan mejor con el precio que las variables crudas.

In [ ]:
# rooms_per_house: habitaciones por vivienda (indica tamaño promedio de los hogares)
df['rooms_per_house'] = df['total_rooms'] / df['households']

# people_per_house: personas por vivienda (indica densidad de ocupación)
df['people_per_house'] = df['population'] / df['households']

# bedrooms_ratio: proporción de dormitorios sobre habitaciones totales (indica diseño de la vivienda)
df['bedrooms_ratio'] = df['total_bedrooms'] / df['total_rooms']

# Verificar que las nuevas columnas se crearon correctamente
df[['rooms_per_house', 'people_per_house', 'bedrooms_ratio']].describe()

**Hallazgo:** los ratios permiten ver patrones que las variables crudas ocultan:
- `rooms_per_house` promedio ~5.4 (casas típicas de 5-6 ambientes)
- `people_per_house` promedio ~2.9 (familias de 2-3 personas)
- `bedrooms_ratio` promedio ~0.21 (~21% de las habitaciones son dormitorios)